# ResNet — Paper Result Reproduction (pretrained weights, NO training)

Ei notebook paper-er **ResNet** model-ta hubohu reproduce kore:

- **Architecture**: repo-r `src/tvt_models.py`-er `Resnet()` theke exact copy (64-ta stacked
  residual block, 469,393 parameter)
- **Weights**: `inf_model_007_256_resnet.h5` -- paper-er **pretrained** checkpoint.
  **Kono training nei**, shudhu load kore inference।
- **Data**: tomar Kaggle dataset-er `test_data.npz` (paper Figs. 5-6 condition:
  L=3, P=Q=16, nt=nr=16, SNR -10..25 dB)
- **Metric**: repo-r `TVT_Blob_Inference.py`-er hubohu same logic
- **Output**: paper-er motoi RMSE-vs-SNR o Pd-vs-SNR plot, ar table

Training nei mane protibar cholaate matro kayek minute lagbe -- experiment korar
jonno eta-i base।

---

## Cholaanor age: weights file ta lagbe

`inf_model_007_256_resnet.h5` (2.8 MB) repo-te ache. Duivabe pete paro:

**1. Kaggle dataset-e upload kore dao** (shobcheye shoja) -- tarpor ei notebook
nije-i khuje nebe.

**2. Ba notebook-e clone koro:**
```
!git clone --depth 1 https://github.com/Mishatmilon059/DL_DOA_CLONE.git /kaggle/working/repo
```
Tarpor path hobe `/kaggle/working/repo/DL_DOA/models/inf_model_007_256_resnet.h5`

Niche Part 2-er cell duita jaygatei khuje dekhe.

## Part 0 — Setup + imports

In [ ]:
import importlib, subprocess, sys

def ensure(pip_name, import_name=None):
    import_name = import_name or pip_name
    try:
        importlib.import_module(import_name)
        print(f'{import_name}: already available, skip.')
    except ImportError:
        print(f'{import_name}: not found, installing {pip_name} ...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pip_name], check=True)

for pip_name, import_name in [('opencv-python-headless', 'cv2')]:
    ensure(pip_name, import_name)
print('Dependency check done.')

In [ ]:
import os, pickle, time, itertools

import numpy as np
from scipy.optimize import linear_sum_assignment
import matplotlib.pyplot as plt
import cv2

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
import tensorflow as tf
from tensorflow.keras.layers import Conv2D, Input, BatchNormalization, Add
from tensorflow.keras import Model

np.random.seed(42)
tf.random.set_seed(42)

print('TensorFlow:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

## Part 1 — ResNet architecture (repo-r `tvt_models.py` theke hubohu)

Paper-er design: input 64x64x2 -> ekta Conv2DTranspose diye 128x128 -> **64-ta stacked
residual block** (protita-te duita 5x5 conv + BatchNorm + skip connection) -> ar ekta
Conv2DTranspose diye 256x256x1 output.

Ekta line-o bodlano hoy ni -- weights load korte hole architecture hubohu milte hobe.

In [ ]:
def res_conv(x, filters=12):
    """Basic residual block: two 5x5 Conv2D + BatchNorm, then skip-add."""
    x_skip = x
    f1 = filters
    x = Conv2D(f1, kernel_size=(5, 5), strides=(1, 1), padding='same')(x)
    x = BatchNormalization()(x)
    x = tf.keras.layers.Activation('relu')(x)
    x = Conv2D(f1, kernel_size=(5, 5), strides=(1, 1), padding='same')(x)
    x = BatchNormalization()(x)
    x = Add()([x, x_skip])
    x = tf.keras.layers.Activation('relu')(x)
    return x


def Resnet(input_shape=(64, 64, 2), output_dim=1):
    """Paper-er ResNet: 64 stacked residual blocks, 64x64x2 -> 256x256x1."""
    x_in = Input(shape=input_shape)
    x = tf.keras.layers.Conv2DTranspose(12, (5, 5), strides=(2, 2), padding='same',
                                        output_padding=None, data_format=None,
                                        dilation_rate=(1, 1), activation=None)(x_in)
    for i in range(64):
        x = res_conv(x)
    x = tf.keras.layers.Conv2DTranspose(output_dim, (5, 5), strides=(2, 2), padding='same',
                                        output_padding=None, data_format=None,
                                        dilation_rate=(1, 1), activation=None)(x)
    return Model(inputs=x_in, outputs=x)


resnet = Resnet(input_shape=(64, 64, 2))
print('ResNet parameters:', f'{resnet.count_params():,}')
print('  (expected: 469,393 -- ei songkha na mille architecture mismatch)')

## Part 2 — Pretrained weights o dataset khujo

Duita jinish lagbe: weights file (`inf_model_007_256_resnet.h5`) ar tomar test set।
Ei cell `/kaggle/input`, `/kaggle/working`, `/content` ar current dir -- shob jaygay
recursively khuje dekhe, tai kono path hardcode korte hobe na।

In [ ]:
SEARCH_ROOTS = ['/kaggle/input', '/kaggle/working', '/content', '.']

def find_file(filename, roots=SEARCH_ROOTS):
    for root in roots:
        if not os.path.isdir(root):
            continue
        for dirpath, dirnames, filenames in os.walk(root):
            if filename in filenames:
                return os.path.join(dirpath, filename)
    return None

def find_dataset_dir(roots=SEARCH_ROOTS, markers=('test_data.npz',)):
    for root in roots:
        if not os.path.isdir(root):
            continue
        for dirpath, dirnames, filenames in os.walk(root):
            if any(m in filenames for m in markers):
                return dirpath
    return None

WEIGHTS_PATH = find_file('inf_model_007_256_resnet.h5')
DATA_DIR = find_dataset_dir()

print('WEIGHTS_PATH =', WEIGHTS_PATH)
print('DATA_DIR     =', DATA_DIR)

assert WEIGHTS_PATH is not None, (
    'inf_model_007_256_resnet.h5 paoa jayni. Hoy Kaggle dataset-e upload koro, '
    'noyto ekta cell-e ei repo clone koro:\n'
    '  !git clone --depth 1 https://github.com/Mishatmilon059/DL_DOA_CLONE.git /kaggle/working/repo'
)
assert DATA_DIR is not None, 'test_data.npz paoa jayni -- Kaggle dataset attach kora ache to?'

### Part 2.1 — Weights load koro (**training nei**)

In [ ]:
resnet.load_weights(WEIGHTS_PATH)
print('Pretrained weights loaded from:', WEIGHTS_PATH)
print('NOTE: kono training hocche na -- eta paper-er nijer checkpoint.')

# quick smoke test: output shape thik ache kina
_probe = resnet(np.zeros((1, 64, 64, 2), dtype=np.float32), training=False).numpy()
print('output shape:', _probe.shape, '(expected (1, 256, 256, 1))')

## Part 3 — Test set load koro

Paper-er Figs. 5-6 condition: **L=3, P=Q=16, nt=nr=16, SNR -10 theke 25 dB**।
ResNet shorashori dataset-e rakha 64x64x2 input ney, tai kono conversion lage na।

In [ ]:
def _p(*names):
    for n in names:
        q = os.path.join(DATA_DIR, n)
        if os.path.exists(q):
            return q
    return None

def _load_features(path):
    if path is None:
        return None
    if path.endswith('.pkl'):
        with open(path, 'rb') as f:
            return pickle.load(f)
    return np.load(path, allow_pickle=True)

X_test = np.load(_p('test_data.npz'))['data']
Y_test = np.load(_p('test_gt.npz'))['data']
meta_test = np.load(_p('test_meta.npz'))['data']
feat_test = _load_features(_p('test_features.npy', 'test_features.pkl'))

print('X_test   :', X_test.shape)
print('Y_test   :', Y_test.shape)
print('meta_test:', meta_test.shape)
print('features :', len(feat_test))
print()
print('conditions in the test set:')
print('  L values  :', np.unique(meta_test[:, 0]).astype(int))
print('  SNR values:', np.unique(meta_test[:, 1]).astype(int))
print('  P values  :', np.unique(meta_test[:, 2]).astype(int))

## Part 4 — Inference utilities (repo-r `TVT_Blob_Inference.py` theke hubohu)

Heatmap -> blob detect -> pixel theke angle -> Hungarian matching -> RMSE/Pd।
Ekdom repo-r same code, jate number gulo reproduce hoy।

In [ ]:
def prepare_prediction_for_peaks(prediction):
    m = prediction[:, :, 0].numpy() if hasattr(prediction, 'numpy') else prediction[:, :, 0]
    return cv2.normalize(m, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)

def get_blob_detector():
    p = cv2.SimpleBlobDetector_Params()
    p.filterByColor = True; p.blobColor = 255
    p.minThreshold = 0; p.maxThreshold = 255
    p.filterByArea = True; p.minArea = 1; p.maxArea = 1000
    p.filterByCircularity = False; p.filterByConvexity = False; p.filterByInertia = False
    return cv2.SimpleBlobDetector_create(p)

detector = get_blob_detector()

def reorder_keypoints(keypoints, img_norm):
    coords = np.array([kp.pt for kp in keypoints])
    if len(coords) == 0:
        return [], np.array([])
    cr = np.round(coords).astype(int)
    amps = []
    for (x, y) in cr:
        amps.append(img_norm[y, x] if (0 <= y < img_norm.shape[0] and 0 <= x < img_norm.shape[1]) else 0)
    amps = np.array(amps); order = np.argsort(-amps)
    return [keypoints[i] for i in order], amps[order]

def get_blob_peaks(pred, detector):
    img = prepare_prediction_for_peaks(pred)
    kps = detector.detect(img)
    kps, amps = reorder_keypoints(kps, img)
    peaks = np.array([kp.pt for kp in kps]) if len(kps) else np.zeros((0, 2))
    return peaks, amps

def wrap_2pi_to_minus_pi(a):
    a = np.asarray(a)
    return np.where(a > np.pi, a - 2 * np.pi, a)

def peaks_to_angles(peaks, margin_factor=3.0, sigma=0.07, grid_size=256):
    if peaks.shape[0] == 0:
        return np.array([]), np.array([])
    margin = margin_factor * sigma
    pxy = peaks.T
    ext = 2 * np.pi + 2 * margin
    fe = -margin + (pxy / grid_size) * ext
    fm = wrap_2pi_to_minus_pi(fe)
    return np.arccos(-fm[1] / np.pi), np.arccos(fm[0] / np.pi)

def permute_pairs(A, B):
    A = np.asarray(A); B = np.asarray(B)
    d = np.linalg.norm(A[:, None, :] - B[None, :, :], axis=2)
    r, c = linear_sum_assignment(d)
    return [(tuple(A[i]), tuple(B[j])) for i, j in zip(r, c)]

def prepare_for_metric(angles_est, feat):
    Lp = feat.shape[-1]
    if len(angles_est[0]) < Lp:
        return (np.array([feat[0], feat[1]]),
                np.array([np.full((Lp,), np.nan), np.full((Lp,), np.nan)]))
    ae = (angles_est[0][:Lp], angles_est[1][:Lp])
    perm = permute_pairs(list(zip(feat[0], feat[1])), list(zip(ae[0], ae[1])))
    psi_t, phi_t = zip(*[p[0] for p in perm])
    psi_e, phi_e = zip(*[p[1] for p in perm])
    return np.array([psi_t, phi_t]), np.array([psi_e, phi_e])

def get_ang_difference(gt_angles, pred_angles):
    H = np.angle(np.exp(1j * gt_angles) * np.exp(-1j * pred_angles))
    return (H * (180 / np.pi)).flatten()

def filter_angles(d, max_deg_error=1.0):
    return d[np.abs(d) <= max_deg_error], d[np.abs(d) > max_deg_error]

print('Inference utilities ready (identical to the repo).')

## Part 5 — Ekta example: input -> prediction -> detected peaks

In [ ]:
def show_example(idx):
    pred = tf.squeeze(resnet(tf.expand_dims(X_test[idx], 0), training=False), axis=0)
    Lp = feat_test[idx].shape[-1]
    peaks, amps = get_blob_peaks(pred, detector)
    peaks_s = peaks[np.argsort(-amps)[:Lp]] if len(peaks) else peaks

    gt = Y_test[idx, :, :, 0]
    fig, axs = plt.subplots(1, 3, figsize=(13, 4))
    axs[0].imshow(X_test[idx][:, :, 0], cmap='viridis'); axs[0].set_title('input (64x64, real part)')
    axs[1].imshow(gt / max(gt.max(), 1e-9), cmap='hot'); axs[1].set_title('ground truth')
    axs[2].imshow(pred.numpy()[:, :, 0], cmap='hot'); axs[2].set_title('ResNet prediction')
    if len(peaks_s):
        axs[2].scatter(peaks_s[:, 0], peaks_s[:, 1], c='cyan', marker='x', s=80)
    for a in axs:
        a.set_xticks([]); a.set_yticks([])
    plt.suptitle(f'sample {idx}  (L={int(meta_test[idx,0])}, SNR={int(meta_test[idx,1])} dB)')
    plt.tight_layout(); plt.show()

show_example(0)
show_example(len(X_test) // 2)

## Part 6 — Full evaluation (paper-er protocol)

**Guruttopurno detail**: repo-r nijer `run_inference_and_metrics_resnet()`-e jei sample-e
blob detector L-er kom peak paay, shei sample-take **baad deya hoy** (`continue`),
byortho hishebe gona hoy na। Reference number gulo ঐ vabe-i toiri, tai ekhane-o hubohu
shei behavior rakha hoyeche -- na hole number milbe na।

In [ ]:
def run_inference_and_metrics(model, X, meta, feat, batch=64, skip_nan=True):
    preds = model.predict(X, batch_size=batch, verbose=1)
    results, n_skipped = {}, 0

    for i in range(len(X)):
        Lp, SNR, QP = int(meta[i, 0]), int(meta[i, 1]), int(meta[i, 2])
        peaks, amps = get_blob_peaks(preds[i], detector)
        peaks = peaks[np.argsort(-amps)[:Lp]] if len(peaks) else peaks
        ang = peaks_to_angles(peaks, sigma=0.07, grid_size=preds.shape[1])
        gt_a, pr_a = prepare_for_metric(ang, feat[i])
        results.setdefault((Lp, SNR, QP), []).append((gt_a, pr_a))

    final_RMSE, final_Pd = {}, {}
    for cond, examples in results.items():
        good_all, bad_all = [], []
        for gt_a, pr_a in examples:
            if np.isnan(pr_a).any():
                if skip_nan:
                    n_skipped += 1
                    continue                      # repo-r nijer behavior
                bad_all.append(np.full(gt_a.size, 999.0))
                continue
            g, b = filter_angles(get_ang_difference(gt_a, pr_a), 1.0)
            good_all.append(g); bad_all.append(b)
        good_all = np.concatenate(good_all) if good_all else np.array([])
        bad_all = np.concatenate(bad_all) if bad_all else np.array([])
        tot = len(good_all) + len(bad_all)
        final_RMSE[cond] = np.sqrt(np.mean(good_all ** 2)) if len(good_all) else np.nan
        final_Pd[cond] = len(good_all) / tot if tot else np.nan
    return final_RMSE, final_Pd, n_skipped


t0 = time.time()
res_rmse, res_pd, n_skip = run_inference_and_metrics(resnet, X_test, meta_test, feat_test)
print(f'\nevaluated {len(X_test)} samples in {time.time()-t0:.0f}s')
print(f'skipped (fewer than L blobs found): {n_skip}')
print()
for c in sorted(res_rmse, key=lambda k: k[1]):
    print(f'  L={c[0]}  SNR={c[1]:>4}  P={c[2]}   RMSE={res_rmse[c]:.4f}   Pd={res_pd[c]:.4f}')

## Part 7 — Repo-r committed reference number-er shathe milao

Ei number gulo repo-r `DL_DOA/figures_resnet/*.pkl` theke -- ei-i ঐ **verified
reproduction**। Tomar number gulo ei table-er kachakachi ashle bujhbe reproduction
shofol।

In [ ]:
RESNET_REF_RMSE = {(3,-10,16):0.5532, (3,-5,16):0.5118, (3,0,16):0.4581, (3,5,16):0.3920,
                   (3,10,16):0.3256, (3,15,16):0.2792, (3,20,16):0.2528, (3,25,16):0.2377}
RESNET_REF_PD   = {(3,-10,16):0.2043, (3,-5,16):0.4366, (3,0,16):0.6396, (3,5,16):0.7790,
                   (3,10,16):0.8623, (3,15,16):0.8987, (3,20,16):0.9250, (3,25,16):0.9378}

print('='*76)
print(f'{"SNR":>5} | {"RMSE now":>9} {"RMSE ref":>9} {"diff":>8} | {"Pd now":>8} {"Pd ref":>8} {"diff":>8}')
print('-'*76)
rmse_err, pd_err = [], []
for c in sorted(res_rmse, key=lambda k: k[1]):
    if c not in RESNET_REF_RMSE:
        continue
    dr = res_rmse[c] - RESNET_REF_RMSE[c]
    dp = res_pd[c] - RESNET_REF_PD[c]
    rmse_err.append(abs(dr)); pd_err.append(abs(dp))
    print(f'{c[1]:>5} | {res_rmse[c]:>9.4f} {RESNET_REF_RMSE[c]:>9.4f} {dr:>+8.4f} '
          f'| {res_pd[c]:>8.4f} {RESNET_REF_PD[c]:>8.4f} {dp:>+8.4f}')
print('='*76)
print(f'mean |RMSE difference| = {np.mean(rmse_err):.4f}')
print(f'mean |Pd difference|   = {np.mean(pd_err):.4f}')
print()
if np.mean(pd_err) < 0.05 and np.mean(rmse_err) < 0.05:
    print('REPRODUCTION: PASS -- number gulo reference-er shathe mile geche.')
else:
    print('REPRODUCTION: difference ache. Shombhabbo karon:')
    print('  - tomar test set-e proti condition-e sample shongkha kom (statistical noise)')
    print('  - dataset alada seed diye generate kora')
    print('  (reference gulo proti condition-e 1000 sample diye kora)')

## Part 8 — Paper-er motoi plot (Figs. 5-6)

In [ ]:
snrs = sorted(s for (l, s, q) in res_rmse if l == 3 and q == 16)
rmse_now = [res_rmse[(3, s, 16)] for s in snrs]
pd_now = [res_pd[(3, s, 16)] for s in snrs]
rmse_ref = [RESNET_REF_RMSE[(3, s, 16)] for s in snrs]
pd_ref = [RESNET_REF_PD[(3, s, 16)] for s in snrs]

fig, axs = plt.subplots(1, 2, figsize=(13, 4.5))

axs[0].plot(snrs, rmse_ref, 's--', color='gray', label='ResNet (repo reference)', alpha=0.7)
axs[0].plot(snrs, rmse_now, 'o-', color='crimson', label='ResNet (this run)')
axs[0].set_xlabel('SNR (dB)'); axs[0].set_ylabel('RMSE (deg)')
axs[0].set_title('RMSE vs SNR  (paper Fig. 5)')
axs[0].legend(); axs[0].grid(alpha=0.3)

axs[1].plot(snrs, pd_ref, 's--', color='gray', label='ResNet (repo reference)', alpha=0.7)
axs[1].plot(snrs, pd_now, 'o-', color='crimson', label='ResNet (this run)')
axs[1].set_xlabel('SNR (dB)'); axs[1].set_ylabel('Probability of detection')
axs[1].set_ylim(-0.02, 1.02)
axs[1].set_title('Pd vs SNR  (paper Fig. 6)')
axs[1].legend(); axs[1].grid(alpha=0.3)

plt.tight_layout(); plt.show()

### Part 8.1 — Table (paper-er motoi) + CSV save

In [ ]:
import csv

rows = []
print('='*62)
print(f'{"L":>3} {"SNR (dB)":>9} {"P=Q":>5} {"RMSE (deg)":>12} {"Pd":>9}')
print('-'*62)
for s in snrs:
    c = (3, s, 16)
    rows.append({'L': 3, 'SNR_dB': s, 'P': 16,
                 'RMSE_deg': round(float(res_rmse[c]), 4),
                 'Pd': round(float(res_pd[c]), 4)})
    print(f'{3:>3} {s:>9} {16:>5} {res_rmse[c]:>12.4f} {res_pd[c]:>9.4f}')
print('='*62)

out_csv = 'resnet_reproduction_results.csv'
with open(out_csv, 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=['L', 'SNR_dB', 'P', 'RMSE_deg', 'Pd'])
    w.writeheader(); w.writerows(rows)
print(f'\nSaved -> {out_csv}  (Kaggle-er Output tab theke download koro)')

# pickle-o save kora hocche, repo-r figures_resnet/*.pkl-er same format-e
with open('resnet_rmse.pkl', 'wb') as f:
    pickle.dump(res_rmse, f)
with open('resnet_pd.pkl', 'wb') as f:
    pickle.dump(res_pd, f)
print('Saved -> resnet_rmse.pkl, resnet_pd.pkl (repo-r figures_resnet/ format)')

## Part 9 — Ekhon ki experiment korte paro

Ei notebook-e **kono training nei**, tai protibar cholaate matro kayek minute। Ekhon
model-take base hishebe niye onek kichu jachai kora jay:

- **Onno condition**: `test_data.npz`-e jodi onno L ba P thake, shegulo-o evaluate koro
  (paper-er Fig. 8: L = 1..6)
- **Threshold bodle dekho**: `filter_angles(..., max_deg_error=1.0)` -- 0.5 ba 2.0 die
  dekho Pd kivabe bodlay
- **Hardware impairment**: input-e phase error dhukiye robustness dekho (paper Section IV-D)
- **skip_nan=False**: `run_inference_and_metrics(..., skip_nan=False)` dile jei sample-e
  L-er kom blob paoa jay shegulo byortho hishebe gona hobe -- eta ekta **kothin** metric,
  ar ei shongkha-tai onno model-er shathe tulonar jonno beshi shot

In [ ]:
# Extra: stricter metric (blob kom paoa sample gulo byortho hishebe gona)
rmse_strict, pd_strict, _ = run_inference_and_metrics(
    resnet, X_test, meta_test, feat_test, skip_nan=False)

print()
print(f'{"SNR":>5} | {"Pd (repo, skip)":>16} {"Pd (strict)":>13}')
print('-'*40)
for s in snrs:
    c = (3, s, 16)
    print(f'{s:>5} | {res_pd[c]:>16.4f} {pd_strict[c]:>13.4f}')
print()
print('Duitar parthokko = joto shotangsho sample-e L-ta blob-i paoa jay ni.')